## Introduction

The objective of this notebook is to **compare several simple long-only investment strategies:**
1. Equity buy-and-hold
2. Equity–bond buy-and-hold
3. Volatility-driven equity-to-bond switching
4. Drawdown-driven equity-to-bond switching

## Import librairies

In [350]:
import numpy as np
import pandas as pd
import plotly.express as px

## Inputs

In [351]:
eq_asset = 'EXW1.DE'
bond_asset = 'IBGS.MI'
vol_threshold_buy = 0.9
vol_threshold_sell = 0.04
vol_window = 5
dd_threshold_buy = 0.70
dd_threshold_sell = 1.0


## Script

In [352]:
df_eq = pd.read_csv(f'data/{eq_asset}.csv', index_col=0, parse_dates=True)
df_bond = pd.read_csv(f'data/{bond_asset}.csv', index_col=0, parse_dates=True)

## 1. Equity buy-and-hold

In [353]:
sharpe = (df_eq.price.pct_change().mean() * 252) / (df_eq.price.pct_change().std() * np.sqrt(252))
print(f'sharpe ratio: {sharpe:.2f}')

total_performance = df_eq.price.iloc[-1] / df_eq.price.iloc[0]
n = ((df_eq.index[-1]-df_eq.index[0]).days / 365)
x = total_performance**(1/n) -1
print(f'annualized return: {x:.2%}')

display(px.line(df_eq, y='price', title=f'{eq_asset} Price History'))
px.bar(df_eq.groupby(df_eq.index.year).price.last().pct_change(), title=f'{eq_asset} Annual Returns')

sharpe ratio: 0.28
annualized return: 3.82%


## 2. Equity–bond buy-and-hold

In [354]:
df_price = pd.concat([df_eq.price, df_bond.price], axis=1, keys=['equity', 'bond']).dropna()
returns = df_price.pct_change().dropna()
mu = returns.mean() * 252
sigma = returns.cov() * 252
inv_sigma = np.linalg.inv(sigma)
z = np.dot(inv_sigma, mu)
w_opt = z / np.sum(z)
print(f'Optimal Weights:\nEquity: {w_opt[0]:.2%}\nBond: {w_opt[1]:.2%}')
portfolio_returns = returns.dot(w_opt)
portfolio_level = portfolio_returns.add(1).cumprod()
sharpe = (portfolio_returns.mean() * 252) / (portfolio_returns.std() * np.sqrt(252))
print(f'sharpe ratio: {sharpe:.2f}')

total_performance = portfolio_level.iloc[-1] / portfolio_level.iloc[0]
n = ((portfolio_level.index[-1]-portfolio_level.index[0]).days / 365)
x = total_performance**(1/n) -1
print(f'annualized return: {x:.2%}')

display(px.line(portfolio_returns.add(1).cumprod(), title='Optimal Portfolio Growth'))
px.bar(portfolio_level.groupby(portfolio_level.index.year).last().pct_change(), title=f'Portfolio Annual Returns')

Optimal Weights:
Equity: 6.34%
Bond: 93.66%
sharpe ratio: 0.45
annualized return: 1.09%


## 3. Volatility-driven equity-to-bond switching

In [355]:
df_vol = df_price.pct_change().rolling(window=vol_window).std() * np.sqrt(252)
df_signal = pd.DataFrame(index=df_price.index, columns=df_price.columns, dtype=float)
df_signal.loc[df_vol.equity > vol_threshold_buy, 'equity'] = 1
df_signal.loc[df_vol.equity < vol_threshold_sell, 'equity'] = 0
df_signal = df_signal.ffill()
df_signal.loc[:, 'equity'] = df_signal.equity.fillna(0)
df_signal.loc[df_signal.equity == 0, 'bond'] = 1
df_signal.loc[df_signal.equity == 1, 'bond'] = 0
portfolio_level = df_price.pct_change().multiply(df_signal).sum(axis=1).add(1).cumprod()
portfolio_returns = portfolio_level.pct_change().dropna()

sharpe = (portfolio_returns.mean() * 252) / (portfolio_returns.std() * np.sqrt(252))
print(f'sharpe ratio: {sharpe:.2f}')

total_performance = portfolio_level.iloc[-1] / portfolio_level.iloc[0]
n = ((portfolio_level.index[-1]-portfolio_level.index[0]).days / 365)
x = total_performance**(1/n) -1
print(f'annualized return: {x:.2%}')

px.line(portfolio_level, title='Equity Bond Vol Switching Strategy')

sharpe ratio: 0.47
annualized return: 4.64%


## 4. Drawdown-driven equity-to-bond switching

In [356]:
dd = df_price.equity / df_price.equity.expanding().max()
df_signal = pd.DataFrame(index=df_price.index, columns=df_price.columns, dtype=float)
df_signal.loc[dd < dd_threshold_buy, 'equity'] = 1
df_signal.loc[dd >= dd_threshold_sell, 'equity'] = 0
df_signal = df_signal.ffill()
df_signal.loc[:, 'equity'] = df_signal.equity.fillna(0)
df_signal.loc[df_signal.equity == 0, 'bond'] = 1
df_signal.loc[df_signal.equity == 1, 'bond'] = 0
portfolio_level = df_price.pct_change().multiply(df_signal).sum(axis=1).add(1).cumprod()
portfolio_returns = portfolio_level.pct_change().dropna()

sharpe = (portfolio_returns.mean() * 252) / (portfolio_returns.std() * np.sqrt(252))
print(f'sharpe ratio: {sharpe:.2f}')

total_performance = portfolio_level.iloc[-1] / portfolio_level.iloc[0]
n = ((portfolio_level.index[-1]-portfolio_level.index[0]).days / 365)
x = total_performance**(1/n) -1
print(f'annualized return: {x:.2%}')

px.line(portfolio_level, title='Equity Bond Vol Switching Strategy')

sharpe ratio: 0.28
annualized return: 3.64%
